# Evaluating a RAG System

Part of [RAG Lab](https://github.com/sitharaj88/rag-lab). "It feels better" is not a metric. This notebook measures **retrieval quality** with two classic numbers — **Hit Rate@k** and **MRR** (Mean Reciprocal Rank) — on a small labeled set.

Runs on Colab, no API key (uses `sentence-transformers`).

> Companion page: *Advanced → Evaluation* on the RAG Lab site.

In [ ]:
%pip install -q sentence-transformers numpy

## 1. A corpus and a labeled eval set

Each eval item is a question plus the id of the document that *should* be retrieved (the "ground truth"). Building a set like this — even 20–50 items — is the single best investment in a serious RAG project.

In [ ]:
corpus = {
    "d1": "RAG stands for Retrieval-Augmented Generation.",
    "d2": "Embeddings turn text into vectors so similar meanings are close together.",
    "d3": "Chunk overlap keeps ideas that span a boundary retrievable.",
    "d4": "Ollama runs open-source LLMs locally with no API key.",
    "d5": "A vector database finds the nearest embeddings to a query quickly.",
    "d6": "Reranking uses a cross-encoder to reorder retrieved results for relevance.",
}

eval_set = [
    {"question": "What does RAG mean?", "answer_id": "d1"},
    {"question": "How do I run a model locally for free?", "answer_id": "d4"},
    {"question": "Why does overlapping chunks help?", "answer_id": "d3"},
    {"question": "What improves the ordering of search results?", "answer_id": "d6"},
    {"question": "What stores vectors for fast search?", "answer_id": "d5"},
]

## 2. Index the corpus

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_ids = list(corpus.keys())
doc_matrix = embedder.encode([corpus[i] for i in doc_ids], normalize_embeddings=True)

def search(question, top_k=3):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    order = np.argsort(doc_matrix @ q)[::-1][:top_k]
    return [doc_ids[i] for i in order]

## 3. Compute Hit Rate@k and MRR

- **Hit Rate@k**: fraction of questions whose correct document appears in the top *k*.
- **MRR**: average of 1/rank of the correct document (rank 1 → 1.0, rank 2 → 0.5, …). Rewards putting the right answer higher.

In [ ]:
def evaluate(eval_set, k=3):
    hits, rr = 0, 0.0
    for item in eval_set:
        ranked = search(item["question"], top_k=k)
        gold = item["answer_id"]
        if gold in ranked:
            hits += 1
            rr += 1.0 / (ranked.index(gold) + 1)
    n = len(eval_set)
    return {"hit_rate@k": hits / n, "mrr": rr / n, "k": k}

print(evaluate(eval_set, k=3))

## 4. Use it to make a decision

Now metrics can drive choices instead of vibes. Try changing `k`, swapping the embedding model, or editing a document, and watch the numbers move. That feedback loop is how real RAG systems improve.

In [ ]:
for k in (1, 2, 3):
    print(evaluate(eval_set, k=k))

## Next steps

- Grow your eval set to 30–50 real questions from your domain.
- Add **answer-quality** checks (faithfulness, relevance) — see *Advanced → Evaluation* and *Observability* on the RAG Lab site.
- Compare retrieval with and without **reranking**.